# 03 Debug Candidates

Project: **One-Direction**

Inspect generated candidate road segments, top-k candidate recall, distance distribution, and candidate ambiguity.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
candidate_dir = ROOT / 'data/processed/candidates'
report_path = candidate_dir / 'candidate_recall_report.json'
train_path = candidate_dir / 'candidates_train.parquet'
val_path = candidate_dir / 'candidates_val.parquet'
test_path = candidate_dir / 'candidates_test.parquet'

In [ ]:
paths = [p for p in [train_path, val_path, test_path] if p.exists()]
if paths:
    candidates = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
    print(candidates.shape)
    display(candidates.head())
else:
    candidates = None
    print('Run scripts/05_generate_candidates.py first.')

In [ ]:
if report_path.exists():
    print(json.dumps(json.loads(report_path.read_text()), indent=2))

In [ ]:
if candidates is not None:
    display(candidates[['distance_m','yaw_diff_rad','candidate_rank']].describe())
    display(candidates.groupby(['trajectory_id','t']).size().describe())

In [ ]:
if candidates is not None and 'is_gt' in candidates.columns:
    top1 = candidates[candidates['candidate_rank'] == 0].groupby(['trajectory_id','t'])['is_gt'].max().mean()
    top10 = candidates.groupby(['trajectory_id','t'])['is_gt'].max().mean()
    print({'top1_recall': float(top1), 'topk_recall': float(top10)})

In [ ]:
if candidates is not None:
    import matplotlib.pyplot as plt
    candidates['distance_m'].clip(upper=100).hist(bins=50)
    plt.title('Candidate distance distribution')
    plt.xlabel('distance_m')
    plt.ylabel('candidate count')
    plt.show()